# Tutorial 23: Hand Landmarks

This notebook explains, builds, and runs a C++ application that tracks 21 landmarks on each detected hand. It accepts camera or video input and uses a DEEPX NPU for both inference stages.

![Hand Landmarks demo](assets/hand-landmarks-sc.png)

## 1. Processing pipeline

The application contains only the hand pipeline:

```text
Camera or video frame
        |
        v
Palm detector, 192 x 192
        |
        v
Palm box and rotated hand region
        |
        v
Hand crop, 224 x 224 -> landmark model
        |
        v
21 landmarks and handedness -> Qt GUI
```

Palm detection first finds a hand region. Each region is rotated, cropped, and passed to the landmark model. The result includes 21 image-space points, world landmarks, hand presence confidence, and left/right handedness.

## 2. Prerequisites

The system needs a supported DEEPX NPU, the device driver, the DXRT SDK, and a graphical desktop session. Install the C++ build and GUI dependencies with:

```bash
sudo apt update
sudo apt install -y build-essential cmake pkg-config libopencv-dev qtbase5-dev ffmpeg v4l-utils
```

You can check the NPU connection with `dxrt-cli -s`.

In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "app").is_dir() and (cwd / "README.md").is_file():
    PROJECT_DIR = cwd
else:
    PROJECT_DIR = cwd / "notebooks" / "T23-demo-hand-landmarks"

PROJECT_DIR = PROJECT_DIR.resolve()
APP_DIR = PROJECT_DIR / "app"
ASSETS_DIR = PROJECT_DIR / "assets"

assert APP_DIR.is_dir(), f"Application directory not found: {APP_DIR}"
print(f"Project: {PROJECT_DIR}")
print(f"Application: {APP_DIR}")
print(f"Assets: {ASSETS_DIR}")

## 3. Project layout

The application code and scripts are under `app/`. Models and videos stay outside the build tree under `assets/`.

In [ ]:
for path in sorted(PROJECT_DIR.rglob("*")):
    if "build" in path.relative_to(PROJECT_DIR).parts:
        continue
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

## 4. Download and check the resources

The two inference models originate from the [MediaPipe Hand Landmarker](https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker) model bundle provided by Google AI Edge. Its palm-detection and hand-landmark models were converted to DXNN format for DEEPX NPU inference.

`get_resources.sh` downloads the resource archive, extracts the models and sample video into `assets/`, and removes the downloaded archive after successful extraction:

```text
assets/
├── models/
│   ├── hand-detector_192x192.dxnn
│   └── HandLandmarkLite.dxnn
└── videos/
    └── hands.mp4
```

The palm model input must be UINT8 `[1, 192, 192, 3]`. The landmark model input must be UINT8 `[1, 224, 224, 3]`.

In [ ]:
required_resources = [
    ASSETS_DIR / "models" / "hand-detector_192x192.dxnn",
    ASSETS_DIR / "models" / "HandLandmarkLite.dxnn",
    ASSETS_DIR / "videos" / "hands.mp4",
]

for path in required_resources:
    status = "ready" if path.is_file() else "missing"
    print(f"{status:7} {path.relative_to(PROJECT_DIR)}")

missing_resources = [path for path in required_resources if not path.is_file()]

Run the next cell only when resources are missing. Existing files with the same names may be replaced during extraction.

In [ ]:
import subprocess

if missing_resources:
    subprocess.run(
        [str(PROJECT_DIR / "get_resources.sh")],
        cwd=PROJECT_DIR,
        check=True,
    )
else:
    print("All required resources are already available.")

for path in required_resources:
    status = "ready" if path.is_file() else "missing"
    print(f"{status:7} {path.relative_to(PROJECT_DIR)}")

### Optional model inspection

If `dxparse` and the models are available, the next cell prints their tensor information. Otherwise, it skips the check without failing.

In [ ]:
import shutil
import subprocess

dxparse = shutil.which("dxparse")
if dxparse is None:
    print("dxparse is not installed; skipping model inspection.")
else:
    for model_path in required_resources[:2]:
        if not model_path.is_file():
            print(f"Skipping missing model: {model_path.name}")
            continue
        print(f"\n--- {model_path.name} ---")
        subprocess.run([dxparse, "-m", str(model_path), "-v"], check=True)

## 5. C++ code guide

The implementation is in `app/hand_landmarks.cpp`. The next cell locates the main processing sections.

In [ ]:
source_path = APP_DIR / "hand_landmarks.cpp"
source_lines = source_path.read_text(encoding="utf-8").splitlines()
symbols = [
    "struct Options",
    "preprocess_palm_frame",
    "decode_palm_detections",
    "make_landmark_input",
    "run_landmark_async",
    "draw_hand_landmarks",
    "class FrameView",
    "run_detection_loop",
]

for symbol in symbols:
    line_number = next((i for i, line in enumerate(source_lines, 1) if symbol in line), None)
    print(f"{line_number:4}: {symbol}")

### Main implementation stages

1. `Options` and `parse_args` select the input, models, thresholds, display mode, and camera settings.
2. `preprocess_palm_frame` converts BGR to RGB and creates the 192 x 192 palm-detector input. Letterboxing is used by default.
3. `decode_palm_detections` converts raw tensors into palm boxes and seven keypoints, then applies weighted non-maximum suppression.
4. The wrist and middle-finger keypoints define a rotated hand region. `make_landmark_input` warps it to 224 x 224.
5. `run_landmark_async` submits one landmark inference for each detected palm. Its callback converts the results back to frame coordinates.
6. `draw_hand_landmarks` connects and renders the 21 points. `FrameView` displays the frame and performance metrics.
7. `run_detection_loop` joins capture, both inference stages, rendering, optional video saving, and playback pacing.

## 6. Build

`build.sh` configures CMake in Release mode and runs `make` with all available CPU cores. Use `--clean` to remove the previous build directory first.

In [ ]:
subprocess.run(["./build.sh"], cwd=APP_DIR, check=True)

In [ ]:
subprocess.run(["./build/hand_landmarks", "--help"], cwd=APP_DIR, check=True)

## 7. Run with a camera

The script uses camera index 0 at a requested 1280 x 720 and 30 FPS. Set `RUN_CAMERA` to `True` only in a graphical session with the NPU, camera, and models ready. Press `Esc` or `Q` to close the application.

In [ ]:
RUN_CAMERA = False

if RUN_CAMERA:
    subprocess.run([
        "./run_camera.sh", "--camera", "0",
        "--width", "1280", "--height", "720", "--fps", "30",
    ], cwd=APP_DIR, check=True)
else:
    print("Camera execution is disabled. Set RUN_CAMERA = True when the hardware and GUI are ready.")

## 8. Run with a video

`run_video.sh` reads `assets/videos/hands.mp4` and loops it. Set `RUN_VIDEO` to `True` after placing the models and video in the expected locations.

In [ ]:
RUN_VIDEO = False

if RUN_VIDEO:
    subprocess.run(["./run_video.sh"], cwd=APP_DIR, check=True)
else:
    print("Video execution is disabled. Set RUN_VIDEO = True when the resources and GUI are ready.")

## 9. Custom commands

Select another camera and capture mode:

```bash
cd app
./run_camera.sh --camera 2 --width 1920 --height 1080 --fps 30
```

Use a custom video and show palm regions for debugging:

```bash
./build/hand_landmarks --video /path/to/input.mp4 --loop --show-palm
```

Save rendered output:

```bash
./build/hand_landmarks --video /path/to/input.mp4 --save --landmark-only
```

Use `Esc` or `Q` to quit and `F` to toggle full-screen mode.

## 10. Troubleshooting

- If a model is missing, verify both filenames under `assets/models/`.
- If a model is rejected, check its input shape and UINT8 data type with `dxparse`.
- If the camera cannot be opened, use `v4l2-ctl --list-devices` and try another camera index.
- If no window appears, verify that the current session has access to a graphical display.
- If CMake cannot find DXRT, set `DXRT_INSTALLED_DIR` and run `./build.sh --clean`.